# Tanager Mangrove Mapping - 01 Preprocessing

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** HDF5 structure inspection, band extraction, spectral index computation, adaptive threshold calibration per scene.

## 0. Environment Setup

In [ ]:
# Install dependencies (commented out for production)
# !pip install h5py xarray rioxarray rasterio geopandas scipy matplotlib

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import h5py

# ============================================================
# Project root — all paths derived from here
# ============================================================
ROOT       = Path('..').resolve()
DATA_RAW   = ROOT / 'data' / 'raw'
DATA_PROC  = ROOT / 'data' / 'processed'
DATA_AOI   = ROOT / 'data' / 'aoi'
SRC        = ROOT / 'src'

sys.path.insert(0, str(ROOT))
from src.preprocessing import (
    inspect_hdf5,
    load_hdf5,
    hdf5_to_geotiff,
    load_geotiff_bands,
    compute_all_indices,
    compute_water_mask,
    apply_adaptive_threshold,
    RELEVANT_WAVELENGTHS
)

print(f'ROOT         : {ROOT}')
print(f'DATA_RAW     : {DATA_RAW}')
print(f'DATA_PROC    : {DATA_PROC}')

## 1. Scene Inventory

In [ ]:
# ============================================================
# Scene IDs — primary site first
# ============================================================
SCENES = {
    'sangatta'   : '20250302_030003_92_4001',
    'gujarat'    : '20250311_061550_53_4001',
    'elsalvador' : '20250223_165546_32_4001',
    'belize'     : '20250824_171857_84_4001',
    'hcmc'       : '20250407_035527_47_4001',   # confirm coverage first
}

for site, sid in SCENES.items():
    h5_path = DATA_RAW / f'{sid}.h5'
    status  = 'OK' if h5_path.exists() else 'MISSING'
    print(f'  {site:<12}: {status}  ({h5_path.name})')

## 2. HDF5 Structure Inspection — Sangatta

In [ ]:
# ============================================================
# Inspect HDF5 keys, shape, wavelength array
# Run once — confirm structure before running hdf5_to_geotiff()
# ============================================================
SCENE_ID = SCENES['sangatta']
h5_path  = DATA_RAW / f'{SCENE_ID}.h5'

inspect_hdf5(str(h5_path))

# TODO: after inspection, update load_hdf5() in src/preprocessing.py
# to match actual HDF5 structure (REFLECTANCE_PATH, WAVELENGTH_PATH, etc.)

## 3. HDF5 → GeoTIFF Conversion

In [ ]:
# ============================================================
# Convert all scenes — relevant bands only
# Output: data/processed/{scene_id}_{band}_{wl}nm.tif
# Run once — skip if already converted
# ============================================================
print(f'Bands to export: {RELEVANT_WAVELENGTHS}\n')

for site, scene_id in SCENES.items():
    h5_path = DATA_RAW / f'{scene_id}.h5'
    if not h5_path.exists():
        print(f'  {site:<12}: SKIP (HDF5 not found)')
        continue

    print(f'  {site} — converting...')
    output_paths = hdf5_to_geotiff(
        hdf5_path  = str(h5_path),
        output_dir = str(DATA_PROC),
        scene_id   = scene_id
    )
    print(f'  {site:<12}: done\n')

## 4. Spectral Indices — Sangatta

In [ ]:
# ============================================================
# Load processed bands and compute 5 indices
# ============================================================
SCENE_ID = SCENES['sangatta']

data    = load_geotiff_bands(str(DATA_PROC), SCENE_ID)
indices = compute_all_indices(data)

print('\nIndex statistics:')
for name, arr in indices.items():
    print(f'  {name:<8}: min={np.nanmin(arr):.3f}  max={np.nanmax(arr):.3f}  mean={np.nanmean(arr):.3f}')

In [ ]:
# ============================================================
# Diagnostic plot — all 5 indices
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, arr) in zip(axes, indices.items()):
    im = ax.imshow(arr, cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_title(name)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f'Spectral Indices — Sangatta ({SCENE_ID})', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'indices_sangatta.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 5. Adaptive Threshold Calibration — Sangatta

In [ ]:
# ============================================================
# Per-scene bimodal histogram threshold
# Core innovation — calibrated locally, not fixed from literature
# Water mask applied first to keep bimodal peaks clean
# ============================================================
water_mask = compute_water_mask(data)
thresholds = apply_adaptive_threshold(indices, scene_id=SCENE_ID,
                                      water_mask=water_mask)

# Save thresholds for use in 02_classification.ipynb
import json
thresh_path = ROOT / 'outputs' / 'results' / f'thresholds_{SCENE_ID}.json'
with open(thresh_path, 'w') as f:
    json.dump({k: v for k, v in thresholds.items() if v is not None}, f, indent=2)
print(f'\nThresholds saved : {thresh_path}')

In [ ]:
# ============================================================
# Diagnostic plot — histogram + threshold lines for MVI + NDMI
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name in zip(axes, ['MVI', 'NDMI']):
    arr  = indices[name]
    flat = arr[np.isfinite(arr)].ravel()
    ax.hist(flat, bins=256, color='steelblue', alpha=0.7)
    if thresholds.get(name) is not None:
        ax.axvline(thresholds[name], color='red', linewidth=2,
                   label=f'threshold = {thresholds[name]:.4f}')
    ax.set_title(f'{name} — Bimodal Histogram')
    ax.set_xlabel('Index value')
    ax.set_ylabel('Pixel count')
    ax.legend()

plt.suptitle(f'Adaptive Thresholds — Sangatta', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'thresholds_sangatta.png',
            dpi=150, bbox_inches='tight')
plt.show()